# Graph Regression for PyTorch Geometric in TopologicPy
The goal is to predict, as cloase as possible, the **graph label**. The graph label is continous.

## What this notebook assumes

You already have a prepared dataset directory containing:

- `graphs.csv`
- `nodes.csv`
- `edges.csv`

## What this notebook does

1. Reads the dataset from disk
2. Loads the dataset into `topologicpy.PyG`
3. Set the model training hyperparameters
4. Trains a baseline **GraphSAGE** model for **graph regression**
5. Evaluates the model on validation and test graphs
6. Visualizes training history and correlation scatter plot.


## Expected Folder Contents

- `graphs.csv`  
- `nodes.csv`  
- `edges.csv`  
- `meta.yaml`  

---

## This Script

1. Loads the dataset with the PyG helper class  
2. Builds a graph-level GNN model (graph classification by default)  
3. Trains with a train/val split  
4. Evaluates on validation and test sets  
5. Visualizes learning curves, Correlation Scatter Plots, and prints evaluation metrics  

---

## Notes

- **Requires:** `topologicpy >= 0.9.31`, `torch`, `torch_geometric`, `pandas`, `pyyaml`, `numpy`, `plotly`, `scikit-learn`
- **Example Datasets can found at** `https://github.com/wassimj/topologicpy/tree/main/assets/MachineLearning`

### Installation Example

```bash
pip install torch pandas pyyaml numpy plotly scikit-learn
# then install torch-geometric following their official instructions for your OS/CUDA


In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries and utility function

In [ ]:
from __future__ import annotations
import os
import numpy as np
import pandas as pd
from pathlib import Path
import json
import yaml
from topologicpy.PyG import PyG
from topologicpy.Helper import Helper
from topologicpy.Plotly import Plotly

# def pretty_print_metrics(title: str, metrics: dict) -> None:
#     print("\n" + "=" * 80)
#     print(title)
#     print("=" * 80)
#     for k in sorted(metrics.keys()):
#         v = metrics[k]
#         if isinstance(v, float):
#             print(f"{k:30s}: {v:.6f}")
#         else:
#             print(f"{k:30s}: {v}")
#     print("=" * 80 + "\n")

def pretty_print_metrics(title: str, metrics: dict) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    # Convert to DataFrame
    df = pd.DataFrame({
        "metric": list(metrics.keys()),
        "value": list(metrics.values())
    })

    # Sort by metric name (to preserve your original behaviour)
    df = df.sort_values("metric").reset_index(drop=True)

    # Format floats to 6 decimal places
    def _format(v):
        if isinstance(v, float):
            return f"{v:.6f}"
        return v

    df["value"] = df["value"].apply(_format)

    print(df.to_string(index=False))

    print("=" * 80 + "\n")

## 2. Check the TopologicPy Version

In [ ]:
print("The script is compatible with TopologicPy v0.9.31 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## Baseline

```
# ------------------------------------------------------------------
# DATASET LOCATION
# ------------------------------------------------------------------
# Change this path if your prepared clean dataset is elsewhere.
# The folder must contain graphs.csv, nodes.csv, and edges.csv.
DATASET_PATH = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_graph_regression").resolve()

# ------------------------------------------------------------------
# TASK DEFINITION
# ------------------------------------------------------------------
PREDICTION_LEVEL = "graph"              # Prediction target level: graph | node | edge | link.
TASK = "regression"                     # Learning task type: classification | regression | link_prediction.
GRAPH_LABEL_TYPE = "continuous"         # Type of graph-level label: continuous for regression targets such as cooling/heating load.
NODE_LABEL_TYPE = "categorical"         # Type of node-level label: categorical class index, e.g. room type or element type.
EDGE_LABEL_TYPE = "categorical"         # Type of edge-level label: categorical class index; currently not used in this setup.
HOLDOUT_GROUP_BY = "label"              # Grouping key for holdout splitting; keeps items with the same label together where supported.


# ------------------------------------------------------------------
# MODEL DEFINITION
# ------------------------------------------------------------------
CONV = "sage"                           # Graph convolution operator to use; "sage" selects GraphSAGE.
HIDDEN_DIMS = (32, 32)                  # Hidden layer dimensions; two values define two hidden graph-convolution layers.
ACTIVATION = "relu"                     # Activation function used between hidden layers.
DROPOUT = 0.10                          # Dropout probability applied during training to reduce overfitting.
BATCH_NORM = False                      # Whether to apply batch normalisation after hidden layers.
RESIDUAL = False                        # Whether to use residual/skip connections between compatible layers.
POOLING = "mean"                        # Graph-level pooling operation used to aggregate node embeddings into graph embeddings.


# ------------------------------------------------------------------
# DATA SPLIT SETTINGS
# ------------------------------------------------------------------
CROSS_VALIDATION = "holdout"            # Validation strategy; "holdout" uses one train/validation/test split.
TRAIN_RATIO = 0.70                      # Proportion of data assigned to the training set.
VAL_RATIO = 0.15                        # Proportion of data assigned to the validation set.
TEST_RATIO = 0.15                       # Proportion of data assigned to the test set.
RANDOM_STATE = 53                       # Random seed used for reproducible splitting, shuffling, and training initialisation where supported.
SHUFFLE = True                          # Whether to shuffle the dataset before splitting.


# ------------------------------------------------------------------
# TRAINING SETTINGS
# ------------------------------------------------------------------
LEARNING_RATE = 0.001                   # Optimiser learning rate.
BATCH_SIZE = 14                         # Number of graphs/samples per training batch.
EPOCHS = 70                             # Maximum number of training epochs.
OPTIMIZER = "adamw"                     # Optimiser to use for training; AdamW includes decoupled weight decay.
WEIGHT_DECAY = 1e-4                     # L2 regularisation strength used by the optimiser.
GRADIENT_CLIP_NORM = 1.0                # Maximum gradient norm for clipping; helps stabilise training.
EARLY_STOPPING = True                   # Whether to stop training early when validation performance stops improving.
EARLY_STOPPING_PATIENCE = 15            # Number of epochs without validation improvement before early stopping is triggered.
USE_GPU = True                          # Whether to use a CUDA-capable GPU when available.


```

## 4. Specify input parameters

In [ ]:
# ------------------------------------------------------------------
# DATASET LOCATION
# ------------------------------------------------------------------
# Change this path if your prepared clean dataset is elsewhere.
# The folder must contain graphs.csv, nodes.csv, and edges.csv.
DATASET_PATH = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_graph_regression").resolve()

# ------------------------------------------------------------------
# TASK DEFINITION
# ------------------------------------------------------------------
PREDICTION_LEVEL = "graph"              # Prediction target level: graph | node | edge | link.
TASK = "regression"                     # Learning task type: classification | regression | link_prediction.
GRAPH_LABEL_TYPE = "continuous"         # Type of graph-level label: continuous for regression targets such as cooling/heating load.
NODE_LABEL_TYPE = "categorical"         # Type of node-level label: categorical class index, e.g. room type or element type.
EDGE_LABEL_TYPE = "categorical"         # Type of edge-level label: categorical class index; currently not used in this setup.
HOLDOUT_GROUP_BY = "label"              # Grouping key for holdout splitting; keeps items with the same label together where supported.


# ------------------------------------------------------------------
# MODEL DEFINITION
# ------------------------------------------------------------------
CONV = "sage"                           # Graph convolution operator to use; "sage" selects GraphSAGE.
HIDDEN_DIMS = (32, 32)                  # Hidden layer dimensions; two values define two hidden graph-convolution layers.
ACTIVATION = "relu"                     # Activation function used between hidden layers.
DROPOUT = 0.10                          # Dropout probability applied during training to reduce overfitting.
BATCH_NORM = False                      # Whether to apply batch normalisation after hidden layers.
RESIDUAL = False                        # Whether to use residual/skip connections between compatible layers.
POOLING = "mean"                        # Graph-level pooling operation used to aggregate node embeddings into graph embeddings.


# ------------------------------------------------------------------
# DATA SPLIT SETTINGS
# ------------------------------------------------------------------
CROSS_VALIDATION = "holdout"            # Validation strategy; "holdout" uses one train/validation/test split.
TRAIN_RATIO = 0.70                      # Proportion of data assigned to the training set.
VAL_RATIO = 0.15                        # Proportion of data assigned to the validation set.
TEST_RATIO = 0.15                       # Proportion of data assigned to the test set.
RANDOM_STATE = 53                       # Random seed used for reproducible splitting, shuffling, and training initialisation where supported.
SHUFFLE = True                          # Whether to shuffle the dataset before splitting.


# ------------------------------------------------------------------
# TRAINING SETTINGS
# ------------------------------------------------------------------
LEARNING_RATE = 0.001                   # Optimiser learning rate.
BATCH_SIZE = 14                         # Number of graphs/samples per training batch.
EPOCHS = 70                             # Maximum number of training epochs.
OPTIMIZER = "adamw"                     # Optimiser to use for training; AdamW includes decoupled weight decay.
WEIGHT_DECAY = 1e-4                     # L2 regularisation strength used by the optimiser.
GRADIENT_CLIP_NORM = 1.0                # Maximum gradient norm for clipping; helps stabilise training.
EARLY_STOPPING = True                   # Whether to stop training early when validation performance stops improving.
EARLY_STOPPING_PATIENCE = 15            # Number of epochs without validation improvement before early stopping is triggered.
USE_GPU = True                          # Whether to use a CUDA-capable GPU when available.


## 5. Load the CSV Dataset (The Example Has Continous Labels, Task is Graph-level Regression)

In [ ]:
# Optional: read meta.yaml (purely informational)
meta_path = DATASET_PATH / "meta.yaml"
if meta_path.exists():
    meta = yaml.safe_load(meta_path.read_text(encoding="utf-8"))
    print("Loaded meta.yaml:")
    print(json.dumps(meta, indent=2))
else:
    print("meta.yaml not found (this is fine).")

# This dataset has graphs.csv with a categorical 'label' column -> graph classification.
pyg = PyG.ByCSVPath(
    path=str(DATASET_PATH),
    level=PREDICTION_LEVEL,
    task=TASK,
    graphLabelType=GRAPH_LABEL_TYPE,
    nodeLabelType=NODE_LABEL_TYPE,
    edgeLabelType=EDGE_LABEL_TYPE,
    holdout_group_by=HOLDOUT_GROUP_BY,

    # If your headers differ, override here (your attached CSVs match defaults):
    # graphIDHeader="graph_id", graphLabelHeader="label",
    # nodeIDHeader="node_id", nodeLabelHeader="label",
    # edgeSRCHeader="src_id", edgeDSTHeader="dst_id", edgeLabelHeader="label",
)

## 6. Set Hyperparameters

In [ ]:
## Baseline
pyg.SetHyperparameters(
    cv=CROSS_VALIDATION,
    split=(TRAIN_RATIO, VAL_RATIO, TEST_RATIO),
    random_state=RANDOM_STATE,
    shuffle=SHUFFLE,
    holdout_group_by=HOLDOUT_GROUP_BY,

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    optimizer=OPTIMIZER,
    gradient_clip_norm=GRADIENT_CLIP_NORM,
    early_stopping=EARLY_STOPPING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    use_gpu=USE_GPU,

    conv=CONV,
    hidden_dims=HIDDEN_DIMS,
    activation=ACTIVATION,
    dropout=DROPOUT,
    batch_norm=BATCH_NORM,
    residual=RESIDUAL,
    pooling=POOLING,
)


# Print a compact summary of the current config and inferred dims/classes
print("PyG config summary:")
print(pyg.Summary())

## 6. Train the Model

In [ ]:
history = pyg.Train()  # returns dict of per-epoch curves (loss + metrics when available)
print(history)

## 7. Plot the Training and Validation Loss Curves

In [ ]:
fig_hist = pyg.PlotHistory()
fig_hist.update_layout({"height": 700, "width":800})
fig_hist.show()

## 8. Validate the Model

In [ ]:
val_metrics = pyg.Validate()
pretty_print_metrics("Validation metrics", val_metrics)


## 9. Test the Model

In [ ]:
test_metrics = pyg.Test()
pretty_print_metrics("Test metrics", test_metrics)

## 10. Plot the Parity Chart (for Regression Only)

### For training portion

In [ ]:
pred = pyg.Predict(split="train")
actual = np.array(pred["y_true"]).reshape(-1)
predicted = np.array(pred["pred"]).reshape(-1)
fig = Plotly.FigureByCorrelation(
    actual,
    predicted,
    title="Correlation between Actual and Predicted Values",
    xTitle="Actual Values",
    yTitle="Predicted Values",
    showIdentity = True,
    showBestFit=True,
    dotSize=6,
    dotColor="blue",
    lineColor="purple",
    width=900,
    height=800,
    theme='default',
    backgroundColor="lightgrey",
)

Plotly.Show(fig)

### For validation portion

In [ ]:
pred = pyg.Predict(split="val")
actual = np.array(pred["y_true"]).reshape(-1)
predicted = np.array(pred["pred"]).reshape(-1)
fig = Plotly.FigureByCorrelation(
    actual,
    predicted,
    title="Correlation between Actual and Predicted Values",
    xTitle="Actual Values",
    yTitle="Predicted Values",
    showIdentity = True,
    showBestFit=True,
    dotSize=6,
    dotColor="blue",
    lineColor="purple",
    width=900,
    height=800,
    theme='default',
    backgroundColor="lightgrey",
)

Plotly.Show(fig)

### For testing portion

In [ ]:
pred = pyg.Predict(split="test")
actual = np.array(pred["y_true"]).reshape(-1)
predicted = np.array(pred["pred"]).reshape(-1)
fig = Plotly.FigureByCorrelation(
    actual,
    predicted,
    title="Correlation between Actual and Predicted Values",
    xTitle="Actual Values",
    yTitle="Predicted Values",
    showIdentity = True,
    showBestFit=True,
    dotSize=6,
    dotColor="blue",
    lineColor="purple",
    width=900,
    height=800,
    theme='default',
    backgroundColor="lightgrey",
)

Plotly.Show(fig)

### For the whole dataset

In [ ]:
pred = pyg.Predict(split="all")
actual = np.array(pred["y_true"]).reshape(-1)
predicted = np.array(pred["pred"]).reshape(-1)
fig = Plotly.FigureByCorrelation(
    actual,
    predicted,
    title="Correlation between Actual and Predicted Values",
    xTitle="Actual Values",
    yTitle="Predicted Values",
    showIdentity = True,
    showBestFit=True,
    dotSize=6,
    dotColor="blue",
    lineColor="purple",
    width=900,
    height=800,
    theme='default',
    backgroundColor="lightgrey",
)

Plotly.Show(fig)